# Aprendizagem de Máquina para Web Developers

## Processamento de Linguagem Natural: da via clássica aos LLMs

**Formato:** aula remota, ambiente **Google Colab**.

Esta aula tem duas partes que espelham a evolução do NLP:

1. **A via clássica** — transformar texto em números com técnicas fundamentais (pré-processamento, Bag-of-Words, TF-IDF) e treinar um classificador. Entender o _porquê_ e o _como_.
2. **A via moderna** — usar **Large Language Models (LLMs)** via API, e a ponte entre os dois mundos: **embeddings**.

**Objetivo:** que você entenda os trade-offs e saiba escolher a ferramenta certa para cada problema — não decorar uma única resposta.


---

## Parte 1 — Fundamentos de NLP

### 1.1 O problema central: palavras viram números

Algoritmos de ML operam sobre **números**, não texto. O desafio do NLP é converter texto não estruturado em uma **representação numérica** (um vetor) que capture significado.

Para quem vem de desenvolvimento web: é parecido com **desserializar** um `JSON` ou `BLOB` do banco para um objeto estruturado. É, no fundo, um problema de **feature engineering** — as features vêm das palavras e da estrutura do texto.

Por que é difícil criar representações **significativas**:

- **Ambiguidade:** "banco" (assento? instituição?).
- **Sinonímia:** "carro" e "automóvel" querem dizer o mesmo.
- **Ordem e contexto:** "o cão mordeu o homem" ≠ "o homem mordeu o cão".
- **Ironia/sarcasmo:** o sentido é o oposto do literal.

A abordagem moderna é **estatística**: o modelo _aprende_ padrões a partir de exemplos, em vez de seguir regras gramaticais escritas à mão.


### 1.2 O pipeline clássico de pré-processamento

Uma "linha de montagem" para limpar e padronizar o texto antes de vetorizar:

`Texto bruto → minúsculas → tokenização → remove pontuação → remove stopwords → vetorização`

Usamos o **NLTK**, biblioteca fundamental de NLP em Python.


In [1]:
# No Colab o NLTK já vem instalado. Baixamos apenas os pacotes de dados.
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Em versões recentes do NLTK o word_tokenize exige 'punkt_tab' (além de 'punkt').
# Baixar os dois evita o erro mais comum nesta aula.
for pacote in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.download(pacote, quiet=True)
    except Exception as e:
        print("aviso:", pacote, e)


def preprocess_text(text):
    """Pipeline de pré-processamento: minúsculas, tokeniza, remove pontuação e stopwords."""
    text_lower = text.lower()
    tokens = word_tokenize(text_lower, language="portuguese")
    # só tokens alfabéticos
    tokens_no_punct = [w for w in tokens if w.isalpha()]
    stop_words = set(stopwords.words("portuguese"))
    filtered = [w for w in tokens_no_punct if w not in stop_words]
    return filtered


raw_text = "Isso é um ÓTIMO exemplo de frase, mostrando a remoção de stopwords em português!"
print("Original: ", raw_text)
print("Processado:", preprocess_text(raw_text))

/home/aldemir/Workspace/repositories/web-academy-ufam/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/aldemir/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


Original:  Isso é um ÓTIMO exemplo de frase, mostrando a remoção de stopwords em português!
Processado: ['ótimo', 'exemplo', 'frase', 'mostrando', 'remoção', 'stopwords', 'português']


**O que cada etapa faz:**

- **Minúsculas:** "Produto", "produto" e "PRODUTO" viram o mesmo token.
- **Tokenização:** quebra o texto em unidades; `word_tokenize` é mais esperto que um `split()` simples.
- **Remover pontuação:** `,`, `.`, `!` costumam ser ruído; `isalpha()` filtra bem.
- **Remover stopwords:** palavras muito comuns ("o", "a", "de", "que") carregam pouco significado distintivo.


### 1.3 Vetorização

#### 1.3.1 Bag-of-Words (BoW): contar frequência de palavras

Representa cada documento pela **contagem** de cada palavra, ignorando ordem e gramática. O resultado é uma **matriz documento-termo**: linhas são documentos, colunas são palavras.


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "O produto é excelente e a entrega do produto foi rápida.",
    "Produto ruim, não recomendo.",
    "Entrega excelente, mas o produto deixou a desejar.",
]

vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(corpus)

df_bow = pd.DataFrame(
    X_bow.toarray(), columns=vectorizer_bow.get_feature_names_out())
print("Matriz Documento-Termo (Bag-of-Words):")
df_bow

Matriz Documento-Termo (Bag-of-Words):


,deixou,desejar,do,entrega,excelente,foi,mas,não,produto,recomendo,ruim,rápida
0,0,0,1,1,1,1,0,0,2,0,0,1
1,0,0,0,0,0,0,0,1,1,1,1,0
2,1,1,0,1,1,0,1,0,1,0,0,0


**Limitação do BoW:** a ordem se perde. "não recomendo" vira dois tokens soltos, e o "não" desgruda do que ele nega.

**Solução parcial — N-gramas:** sequências de _n_ palavras. **Bigramas** (n=2) capturam pares como "não recomendo".


In [3]:
vectorizer_ngram = CountVectorizer(ngram_range=(1, 2))   # unigramas + bigramas
X_ngram = vectorizer_ngram.fit_transform(corpus)
df_ngram = pd.DataFrame(
    X_ngram.toarray(), columns=vectorizer_ngram.get_feature_names_out())

print("Shape da matriz:", df_ngram.shape)
colunas = ["entrega", "excelente", "entrega excelente",
           "não", "recomendo", "não recomendo"]
df_ngram[[col for col in colunas if col in df_ngram.columns]]

Shape da matriz: (3, 26)


,entrega,excelente,entrega excelente,não,recomendo,não recomendo
0,1,1,0,0,0,0
1,0,0,0,1,1,1
2,1,1,1,0,0,0


#### 1.3.2 TF-IDF: dar peso às palavras relevantes

O BoW trata todas as palavras igual. Palavras comuns ("produto") dominam sem informar muito.

O **TF-IDF** calcula um peso que reflete a importância de uma palavra num documento, dentro do corpus todo:

- **TF (Term Frequency):** quão frequente a palavra é _no documento_ (importância local).
- **IDF (Inverse Document Frequency):** quão _rara_ a palavra é _no corpus_ (raridade/informação).

`TF-IDF = TF × IDF`. O peso é alto para palavras frequentes num documento mas raras no geral — as mais distintivas.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(corpus)
df_tfidf = pd.DataFrame(
    X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

print("Matriz Documento-Termo (TF-IDF):")
df_tfidf.round(2)

Matriz Documento-Termo (TF-IDF):


,deixou,desejar,do,entrega,excelente,foi,mas,não,produto,recomendo,ruim,rápida
0,0.00,0.00,0.42,0.32,0.32,0.42,0.00,0.00,0.50,0.00,0.00,0.42
1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.55,0.32,0.55,0.55,0.00
2,0.47,0.47,0.00,0.36,0.36,0.00,0.47,0.00,0.28,0.00,0.00,0.00


**Lendo o resultado:**

- **"produto"** tende ao score mais baixo — aparece em todos os documentos (IDF baixo).
- **"rápida"**, **"ruim"** têm scores altos em suas linhas — são raras (IDF alto) e, portanto, distintivas.

O TF-IDF automaticamente **rebaixa** palavras comuns e **promove** as informativas, gerando features de melhor qualidade para o classificador. É a base da via clássica que você vai usar no exercício.


---

## Parte 2 — A ponte entre os dois mundos: Embeddings

O TF-IDF representa texto contando palavras. Ele **não entende** que "carro" e "automóvel" são próximos — para ele, são colunas diferentes e sem relação.

**Embeddings** resolvem isso. São vetores densos (não mais contagens esparsas) em que **textos com significado parecido ficam próximos no espaço**. "Ótimo produto" e "excelente item" caem perto; "produto horrível" cai longe. É o que hoje sustenta **busca semântica**, sistemas de recomendação e o **RAG** (Retrieval-Augmented Generation) que alimenta chatbots com base em documentos.

Você pode gerar embeddings via API, do mesmo jeito que chamaria um LLM. O padrão de uso:

1. envia um texto → recebe um vetor de números (ex.: 768 ou 1536 dimensões);
2. para comparar dois textos, mede a **similaridade de cosseno** entre os vetores.

O bloco abaixo mostra o conceito **sem depender de API** — usando um modelo local pequeno — para você ver a ideia funcionando. Na prática de produção, trocaria isso por uma chamada à API de embeddings do seu provedor.


In [ ]:
# Demonstração do conceito de embeddings SEM API (modelo local pequeno, roda no Colab).
# Em produção, você trocaria isto por uma chamada à API de embeddings do provedor.
# !pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

modelo = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2")  # suporta português

frases = [
    "o produto é excelente",
    "que item maravilhoso",      # significado parecido com o 1º
    "o produto é horrível",      # significado oposto
]
vetores = modelo.encode(frases)
print("Cada frase virou um vetor de tamanho:", vetores.shape[1])

sim = cosine_similarity(vetores)
print("\nSimilaridade (1.0 = idêntico):")
print(f"  'excelente' vs 'maravilhoso': {sim[0][1]:.2f}   (esperado: ALTO)")
print(f"  'excelente' vs 'horrível'   : {sim[0][2]:.2f}   (esperado: BAIXO)")

/home/aldemir/Workspace/repositories/web-academy-ufam/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5768.41it/s]


Cada frase virou um vetor de tamanho: 384

Similaridade (1.0 = idêntico):
  'excelente' vs 'maravilhoso': 0.77   (esperado: ALTO)
  'excelente' vs 'horrível'   : 0.19   (esperado: BAIXO)


> **A diferença essencial:** o TF-IDF compararia essas frases por **palavras em comum** — e "excelente" e "maravilhoso" não compartilham nenhuma, então ele as veria como distantes. O embedding as vê como **próximas em significado**. É esse salto que separa a via clássica da moderna.


---

## Parte 3 — LLMs (Large Language Models)

### 3.1 O que são

Modelos de deep learning de escala enorme (bilhões a trilhões de parâmetros), pré-treinados em vastas quantidades de texto e código. A arquitetura-chave é o **Transformer**, cujo mecanismo de **auto-atenção (self-attention)** captura contexto de longo alcance. A tarefa de treino — **prever a próxima palavra** — força o modelo a aprender gramática, fatos, semântica e alguma capacidade de raciocínio.

Famílias atuais: **GPT** (OpenAI), **Gemini** (Google), **Claude** (Anthropic), **Llama** (Meta), entre outras.

### 3.2 O que fazem

São modelos de propósito geral: geração de conteúdo, sumarização, tradução, análise de sentimento (com nuance), chatbots, geração e depuração de código — tudo com a mesma base, mudando só o **prompt**.

### 3.3 Como um web developer interage: APIs

Quase todo uso real é via **API REST**. A infraestrutura de GPU fica com o provedor. Seu foco:

1. **gerenciar a chave de API com segurança** (nunca colar no código!);
2. dominar **engenharia de prompt** — escrever instruções claras.

### 3.4 Conceitos que você precisa conhecer antes de chamar a API

- **Token:** LLMs não contam palavras, contam _tokens_ (pedaços de palavra). "inacreditável" pode virar 3-4 tokens. **Você paga por token** (entrada + saída), então prompt e resposta têm custo.
- **Alucinação:** o modelo pode gerar informação **plausível mas falsa**, com total confiança. É o maior risco em produção — nunca confie cegamente na saída para fatos críticos.
- **Não-determinismo:** a mesma pergunta pode dar respostas ligeiramente diferentes. Para tarefas que exigem formato fixo, veja "saída estruturada" na Parte 4.


### 3.5 Chamando a API — os três grandes provedores

**Sobre nomes de modelo:** os nomes de modelo mudam **muito rápido** (a cada poucas semanas surge um novo e outro é aposentado). Por isso este material **não fixa** um nome específico no meio do código — usamos uma variável `MODELO` no topo de cada exemplo e um link para a lista atual do provedor. **Confira o nome vigente antes de rodar.**

**Sobre a chave de API no Colab:** não cole a chave no código. Use o cofre de _Secrets_ do Colab (ícone de chave 🔑 na barra lateral), guarde sua chave lá e leia com `google.colab.userdata`. Assim ela não vaza se você compartilhar o notebook.


In [ ]:
# Como ler a chave de API com segurança no Colab (padrão para todos os exemplos abaixo)
# 1. Clique no ícone de chave (🔑) na barra lateral esquerda do Colab.
# 2. Adicione um secret, por ex. OPENAI_API_KEY, e cole o valor.
# 3. Leia assim:

# from google.colab import userdata
# minha_chave = userdata.get("OPENAI_API_KEY")

print("Configure seus secrets no Colab antes de rodar os exemplos de API.")

#### OpenAI (modelos GPT)

Docs / lista de modelos atuais: https://platform.openai.com/docs/models


In [ ]:
# !pip install -q openai
# from openai import OpenAI
# from google.colab import userdata
#
# client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
# MODELO = "gpt-...”   # confira o nome atual na doc; escolha um modelo pequeno/barato
#
# resposta = client.chat.completions.create(
#     model=MODELO,
#     messages=[
#         {"role": "system", "content": "Você é um assistente prestativo."},
#         {"role": "user", "content": "Explique o que é uma API REST em uma frase."},
#     ],
# )
# print(resposta.choices[0].message.content)
print("Exemplo OpenAI — descomente, configure a chave e escolha o MODELO atual.")

#### Google (modelos Gemini)

O SDK do Google mudou: a biblioteca atual é **`google-genai`** (`from google import genai`) — a antiga `google-generativeai` está descontinuada.

Docs / lista de modelos: https://ai.google.dev/gemini-api/docs/models


In [ ]:
# !pip install -q google-genai
from google import genai
# from google.colab import userdata

# client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODELO = "gemini-3.5-flash-lite"  # confira o nome atual na doc

resposta = client.models.generate_content(
    model=MODELO,
    contents="Explique o que é uma API REST em uma frase.",
)
print(resposta.text)
# print("Exemplo Gemini — descomente, configure a chave e escolha o MODELO atual.")

Uma **API REST** é uma interface que permite que diferentes sistemas comuniquem-se entre si pela internet de forma simples e padronizada, utilizando os mesmos métodos do protocolo HTTP (como GET, POST, PUT e DELETE) que os navegadores usam para carregar páginas web.


#### Anthropic (modelos Claude)

Docs / lista de modelos: https://docs.anthropic.com/en/docs/about-claude/models


In [9]:
# !pip install -q anthropic
# import anthropic
# from google.colab import userdata
#
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
# MODELO = "claude-...”   # confira o nome atual na doc
#
# resposta = client.messages.create(
#     model=MODELO,
#     max_tokens=200,
#     messages=[{"role": "user", "content": "Explique o que é uma API REST em uma frase."}],
# )
# print(resposta.content[0].text)
print("Exemplo Claude — descomente, configure a chave e escolha o MODELO atual.")

Exemplo Claude — descomente, configure a chave e escolha o MODELO atual.


> Repare no padrão comum aos três: você instancia um cliente com a chave, escolhe um modelo, manda mensagens e lê a resposta em texto. Trocar de provedor é trocar poucos detalhes de sintaxe. **Isso é a "abstração de complexidade" das APIs na prática.**


---

## Parte 4 — Saída estruturada (JSON): o que mais importa para web dev

Um LLM devolvendo um parágrafo de texto é difícil de usar num sistema. O que um back-end quer é **JSON** que ele possa parsear direto. Você consegue isso **instruindo o modelo no prompt** a responder apenas em JSON.

Exemplo de prompt para classificação de sentimento que devolve JSON:

```
Analise o sentimento do review abaixo. Responda APENAS com um JSON válido,
sem texto extra, no formato: {"sentimento": "positivo"|"negativo"|"misto"}.

Review: "O design do celular é incrível, mas a bateria dura muito pouco."
```

Resposta esperada do modelo:

```json
{ "sentimento": "misto" }
```

No seu código, você faria `json.loads(resposta)` e usaria o resultado como um dicionário. Muitos provedores hoje têm um **"JSON mode"** ou **structured outputs** que _garante_ JSON válido — veja a doc de cada um. Este é o elo direto entre LLMs e o mundo de APIs em que você já trabalha.

> **Cuidado:** mesmo pedindo JSON, um modelo pode ocasionalmente errar o formato. Em produção, sempre envolva o `json.loads` em `try/except` e trate a falha.


---

## Parte 5 — Duelo de abordagens: NLP clássico vs. API de LLM

**Cenário:** classificar o sentimento de _"O design do celular é incrível, mas a bateria dura muito pouco."_

**Via clássica (TF-IDF + classificador):** coletar dataset rotulado → pré-processar → vetorizar com TF-IDF → treinar `LogisticRegression` → implantar modelo + vetorizador.

**Via moderna (API de LLM):** escrever um prompt (como na Parte 4) → chamar a API → ler a resposta. Sem dataset, sem treino.

### Tabela de trade-offs

| Característica          | Clássica (_build_)                 | Moderna via API (_buy_)           |
| ----------------------- | ---------------------------------- | --------------------------------- |
| **Esforço**             | Alto: dados, treino, deploy        | Baixo: prompt + chamada de API    |
| **Dados necessários**   | Muitos, rotulados                  | Poucos a nenhum (_zero-shot_)     |
| **Custo**               | Inicial alto, por-inferência baixo | Sem custo inicial, paga por token |
| **Flexibilidade**       | Baixa: um modelo, uma tarefa       | Alta: mesma API, várias tarefas   |
| **Interpretabilidade**  | Moderada/alta (caixa branca)       | Baixa (caixa preta)               |
| **Latência**            | Muito baixa (ms)                   | Variável (rede + provedor)        |
| **Manutenção**          | Sua (monitorar, retreinar)         | Do provedor                       |
| **Risco de alucinação** | Não se aplica                      | Presente — exige validação        |

**Conclusão:** é o clássico **controle vs. conveniência**. A melhor escolha costuma ser **híbrida**: modelo clássico (rápido e barato) para os casos simples e de alto volume; LLM para os casos complexos ou raros. Quem entende as duas vias escolhe com critério — que é o objetivo desta aula.


---

## Exercício — Classificador de sentimento para reviews em português

**Objetivo:** aplicar a via clássica para construir, treinar e avaliar um classificador de sentimento em reviews reais em português.

**Dataset:** **B2W-Reviews01** (amostra de 10 mil linhas), o mesmo da aula de deploy — baixa direto por URL, sem login:

```
https://raw.githubusercontent.com/alan-barzilay/NLPortugues/master/Semana%2003/data/b2w-10k.csv
```

Colunas: `review_text` e `overall_rating` (1 a 5).

### Passo a passo

1. **Carregar e explorar:** `pd.read_csv`, depois `.head()`, `.info()`, `.isnull().sum()`. Plote um histograma de `overall_rating`.
2. **Criar o alvo:** nova coluna `sentimento`: notas 4-5 → `1`, notas 1-2 → `0`, **descarte a nota 3**.
3. **Pré-processar:** aplique a função `preprocess_text` da Parte 1 à coluna `review_text`.
4. **Dividir:** `train_test_split` (ex.: 80/20), com `X` = texto e `y` = sentimento.
5. **Vetorizar com TF-IDF:** `.fit()` **só no treino**; `.transform()` em treino e teste.
6. **Treinar:** `LogisticRegression` (ou `LinearSVC`).
7. **Avaliar:** `accuracy_score`, `classification_report` e `confusion_matrix`.

### Desafio bônus — clássico vs. LLM

1. Encontre exemplos em que seu modelo TF-IDF **errou**.
2. Monte um prompt (Parte 4) e peça a uma API de LLM para classificar os mesmos exemplos, pedindo saída em JSON.
3. Compare: o LLM pegou nuances (ironia, "misto") que o TF-IDF perdeu? Houve caso em que o clássico foi melhor? Essa comparação consolida os trade-offs da Parte 5.

> **Continuidade com a próxima aula:** o classificador que você treinar aqui é exatamente o tipo de modelo que, na aula de **Deploy**, vira um microsserviço com FastAPI. Guarde seu código.
